In [ ]:
"""
Enhanced Multi-Dataset Exoplanet Pipeline
- Includes TESS TOI dataset from provided URL
- Robust error handling for all datasets
- Improved data processing
"""

import pandas as pd
import numpy as np
import requests
import os
import joblib
import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# ENHANCED CONFIGURATION
# ==========================================

DATA_DIR = 'exoplanet_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Enhanced NASA Exoplanet Archive queries with TESS TOI
NASA_DATASETS = {
    'cumulative': {
        'url': 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative+where+koi_disposition+is+not+null&format=csv',
        'description': 'Kepler Cumulative KOI table',
        'manual_filename': 'kepler_cumulative.csv'
    },
    'confirmed': {
        'url': 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+ps&format=csv',
        'description': 'Confirmed Planets',
        'manual_filename': 'confirmed_planets.csv'
    },
    'k2pandc': {
        'url': 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+k2pandc&format=csv',
        'description': 'K2 Planets and Candidates',
        'manual_filename': 'k2_planets.csv'
    },
    'tess_toi': {
        'url': 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+toi&format=csv',
        'description': 'TESS Objects of Interest',
        'manual_filename': 'tess_toi.csv'
    }
}

# Enhanced feature mapping with TESS columns
FEATURE_MAPPING = {
    'period': ['koi_period', 'pl_orbper', 'k2_period', 'pl_orbper'],
    'radius': ['koi_prad', 'pl_rade', 'k2_prad', 'pl_rade'],
    'depth': ['koi_depth', 'k2_depth', 'pl_trandep'],
    'duration': ['koi_duration', 'k2_dur', 'pl_trandur'],
    'stellar_temp': ['koi_steff', 'st_teff', 'k2_steff', 'st_teff'],
    'stellar_radius': ['koi_srad', 'st_rad', 'k2_srad', 'st_rad'],
    'stellar_mass': ['koi_smass', 'st_mass', 'k2_smass', 'st_mass'],
    'snr': ['koi_model_snr', 'k2_snr', 'pl_tranmid'],
    'semi_major_axis': ['koi_sma', 'pl_orbsmax', 'k2_sma', 'pl_orbsmax'],
    'inclination': ['koi_incl', 'pl_orbincl', 'k2_incl', 'pl_orbincl'],
    'eccentricity': ['koi_eccen', 'pl_orbeccen', 'k2_eccen', 'pl_orbeccen']
}

# ==========================================
# STEP 1: Enhanced Download Datasets
# ==========================================

def download_nasa_datasets():
    """Download NASA datasets with enhanced error handling and TESS TOI support"""
    print("="*60)
    print("DOWNLOADING NASA EXOPLANET DATASETS")
    print("="*60)

    datasets = {}

    for name, info in NASA_DATASETS.items():
        print(f"\nDownloading {name}...")
        print(f"  Description: {info['description']}")

        try:
            # Special handling for TESS TOI (HTML table)
            if name == 'tess_toi':
                df = download_tess_toi()
                if df is not None:
                    datasets[name] = df
                    # Save to CSV
                    filepath = os.path.join(DATA_DIR, f'{name}.csv')
                    df.to_csv(filepath, index=False)
                    print(f"✓ Downloaded TESS TOI: {len(df)} rows, {len(df.columns)} columns")
                else:
                    print("✗ Failed to download TESS TOI data")
                continue

            # Standard API download for other datasets
            response = requests.get(info['url'], timeout=30)
            response.raise_for_status()

            # Save to CSV
            filepath = os.path.join(DATA_DIR, f'{name}.csv')
            with open(filepath, 'w') as f:
                f.write(response.text)

            # Load into dataframe
            df = pd.read_csv(filepath)
            datasets[name] = df
            print(f"✓ Downloaded {len(df)} rows, {len(df.columns)} columns")

        except requests.exceptions.RequestException as e:
            print(f"✗ Network error: {str(e)}")
            # Try manual file if exists
            manual_file = os.path.join(DATA_DIR, info.get('manual_filename', f'{name}.csv'))
            if os.path.exists(manual_file):
                print(f"  Trying manual file: {manual_file}")
                try:
                    df = pd.read_csv(manual_file)
                    datasets[name] = df
                    print(f"✓ Loaded from manual file: {len(df)} rows")
                except Exception as file_error:
                    print(f"✗ Error loading manual file: {file_error}")
            continue
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            continue

    if not datasets:
        print("\n⚠ No datasets downloaded successfully!")
        print("Trying alternative source...")
        return try_alternative_source()

    return datasets

def download_tess_toi():
    """Special function to download TESS TOI data from HTML table"""
    try:
        print("  Downloading TESS TOI from HTML table...")
        response = requests.get(NASA_DATASETS['tess_toi']['url'], timeout=30)
        response.raise_for_status()

        # Try to read HTML tables
        tables = pd.read_html(response.text)
        if not tables:
            print("  ⚠ No tables found in TESS TOI page")
            return None

        # Usually the first table contains the data
        df = tables[0]
        print(f"  Found {len(tables)} tables, using first one with shape {df.shape}")

        # Clean the dataframe (remove multi-index headers if present)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join(col).strip() for col in df.columns.values]

        return df

    except Exception as e:
        print(f"✗ Error downloading TESS TOI: {str(e)}")
        # Try alternative TESS source
        return try_alternative_tess_source()

def try_alternative_tess_source():
    """Alternative TESS data source"""
    try:
        print("  Trying alternative TESS source...")
        alt_url = "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI?table=toi&select=*"
        df = pd.read_csv(alt_url)
        print(f"✓ Downloaded TESS data from alternative source: {len(df)} rows")
        return df
    except Exception as e:
        print(f"✗ Alternative TESS source failed: {str(e)}")
        return None

def try_alternative_source():
    """Fallback to direct file download"""
    print("\nTrying alternative NASA archive access...")

    alt_urls = [
        'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI?table=cumulative&select=*&format=csv',
        'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI?table=toi&select=*&format=csv'
    ]

    datasets = {}
    for url in alt_urls:
        try:
            df = pd.read_csv(url)
            name = 'cumulative' if 'cumulative' in url else 'tess_toi'
            datasets[name] = df
            print(f"✓ Downloaded {name}: {len(df)} rows")
        except Exception as e:
            print(f"✗ Alternative source failed for {url}: {str(e)}")

    return datasets if datasets else {}

# ==========================================
# STEP 2: Enhanced Standardize Each Dataset
# ==========================================

def standardize_dataset(df, dataset_name):
    """Extract and standardize features with improved label detection for all datasets"""
    print(f"\nProcessing {dataset_name}...")

    standardized = pd.DataFrame()
    standardized['mission'] = dataset_name

    # Map features with safe numeric conversion
    for std_name, possible_cols in FEATURE_MAPPING.items():
        for col in possible_cols:
            if col in df.columns:
                try:
                    standardized[std_name] = pd.to_numeric(df[col], errors='coerce')
                    break
                except Exception as e:
                    print(f"  ⚠ Could not convert {col} to numeric: {e}")
                    continue
        if std_name not in standardized.columns:
            standardized[std_name] = np.nan

    # Enhanced target column detection for all datasets
    target_col = None
    target_mapping = {
        'cumulative': ['koi_disposition'],
        'confirmed': ['pl_name', 'disposition'],  # All are confirmed
        'k2pandc': ['k2_disp', 'disposition'],
        'tess_toi': ['tfopwg_disp', 'disposition', 'TOI', 'TESS Disposition']
    }

    # Try dataset-specific target columns first
    if dataset_name in target_mapping:
        for col in target_mapping[dataset_name]:
            if col in df.columns:
                target_col = col
                break

    # Fallback: search for any disposition column
    if target_col is None:
        for col in df.columns:
            col_lower = col.lower()
            if any(candidate in col_lower for candidate in ['disposition', 'disp', 'status', 'tfopwg']):
                target_col = col
                break

    # Extract labels with enhanced classification
    if target_col and target_col in df.columns:
        standardized['original_label'] = df[target_col].astype(str)

        # Enhanced classification function
        def classify(label):
            if pd.isna(label):
                return np.nan

            label_upper = str(label).upper()

            # TESS-specific classifications
            if dataset_name == 'tess_toi':
                if any(x in label_upper for x in ['CONFIRMED', 'CP', 'KP']):
                    return 1
                elif any(x in label_upper for x in ['FALSE', 'FP', 'FA', 'FALSE POSITIVE']):
                    return 0
                elif 'CANDIDATE' in label_upper:
                    return 0  # Conservative approach
                else:
                    return np.nan

            # General classification for other datasets
            if 'CONFIRMED' in label_upper:
                return 1
            elif any(x in label_upper for x in ['FALSE', 'NOT', 'FALSE POSITIVE']):
                return 0
            elif 'CANDIDATE' in label_upper:
                return 0  # Treat candidates as not confirmed
            else:
                return np.nan

        standardized['target'] = standardized['original_label'].apply(classify)

    # For confirmed planets table, all are confirmed
    elif dataset_name == 'confirmed':
        standardized['original_label'] = 'CONFIRMED'
        standardized['target'] = 1
        print("  Note: All samples are confirmed planets")
    else:
        standardized['target'] = np.nan

    # Remove rows without targets
    initial_len = len(standardized)
    standardized = standardized.dropna(subset=['target'])

    print(f"  Extracted {len(standardized)} samples (removed {initial_len - len(standardized)} without labels)")
    print(f"  Confirmed planets: {int(standardized['target'].sum())}")
    print(f"  Not confirmed: {int((standardized['target'] == 0).sum())}")

    return standardized

# ==========================================
# STEP 3: Combine All Datasets
# ==========================================

def integrate_datasets(datasets):
    """Combine all datasets with enhanced reporting"""
    print("\n" + "="*60)
    print("INTEGRATING DATASETS")
    print("="*60)

    all_data = []

    for name, df in datasets.items():
        print(f"\n--- Processing {name} ---")
        std_df = standardize_dataset(df, name)
        if len(std_df) > 0:
            all_data.append(std_df)
            print(f"✓ Added {len(std_df)} samples from {name}")
        else:
            print(f"⚠ No valid data from {name}")

    if not all_data:
        raise ValueError("No valid data extracted from any dataset!")

    combined = pd.concat(all_data, ignore_index=True)

    print("\n" + "="*60)
    print("INTEGRATION COMPLETE")
    print("="*60)
    print(f"Total samples: {len(combined)}")
    print(f"Confirmed planets: {int(combined['target'].sum())}")
    print(f"Not confirmed: {int((combined['target'] == 0).sum())}")
    print(f"\nMission distribution:")
    mission_counts = combined['mission'].value_counts()
    for mission, count in mission_counts.items():
        print(f"  {mission}: {count} samples")

    # Check for class imbalance
    confirmed_pct = combined['target'].mean() * 100
    print(f"\nClass balance: {confirmed_pct:.1f}% confirmed")

    if confirmed_pct == 0 or confirmed_pct == 100:
        print("⚠ WARNING: Only one class present! Cannot train binary classifier.")
        return None

    if len(combined) < 100:
        print("⚠ WARNING: Very small dataset may affect model performance")

    return combined

# ==========================================
# STEP 4: Data Cleaning
# ==========================================

def clean_data(df, missing_threshold=0.7):
    """Remove features with too much missing data"""
    print("\n" + "="*60)
    print("CLEANING DATA")
    print("="*60)

    # Calculate missing rates
    feature_cols = [col for col in df.columns
                   if col not in ['mission', 'target', 'original_label']]

    missing_rates = df[feature_cols].isna().sum() / len(df)

    print(f"\nMissing data rates:")
    for col in missing_rates.index:
        print(f"  {col}: {missing_rates[col]:.1%}")

    # Keep features below threshold
    keep_features = missing_rates[missing_rates < missing_threshold].index.tolist()

    if not keep_features:
        print("\n⚠ WARNING: No features meet the threshold!")
        print("  Lowering threshold to 0.9...")
        keep_features = missing_rates[missing_rates < 0.9].index.tolist()

    keep_cols = keep_features + ['mission', 'target', 'original_label']
    df_clean = df[keep_cols].copy()

    removed = len(feature_cols) - len(keep_features)
    print(f"\n✓ Removed {removed} features with >{missing_threshold*100}% missing")
    print(f"✓ Kept {len(keep_features)} features for modeling")

    return df_clean

# ==========================================
# STEP 5: Prepare for Machine Learning
# ==========================================

def prepare_for_ml(df, test_size=0.2):
    """Final data preparation with enhanced validation"""
    print("\n" + "="*60)
    print("PREPARING FOR MACHINE LEARNING")
    print("="*60)

    # Separate features and target
    exclude_cols = ['mission', 'target', 'original_label']
    feature_cols = [col for col in df.columns if col not in exclude_cols]

    X = df[feature_cols]
    y = df['target']

    print(f"\nFeatures used: {feature_cols}")

    # Ensure all data is numeric
    X = X.apply(pd.to_numeric, errors='coerce')

    # Impute missing values
    print("\nImputing missing values...")
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(
        imputer.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    # Train-test split with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=test_size, random_state=42, stratify=y
    )

    # Scale features
    print("Scaling features...")
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    print(f"\n✓ Training samples: {len(X_train_scaled)}")
    print(f"✓ Test samples: {len(X_test_scaled)}")
    print(f"✓ Confirmed % (train): {y_train.mean():.1%}")
    print(f"✓ Confirmed % (test): {y_test.mean():.1%}")

    return X_train_scaled, X_test_scaled, y_train, y_test

# ==========================================
# STEP 6: Train Model
# ==========================================

def train_model(X_train, y_train, X_test, y_test):
    """Train and evaluate Random Forest model with enhanced reporting"""
    print("\n" + "="*60)
    print("TRAINING RANDOM FOREST MODEL")
    print("="*60)

    # Check class distribution
    unique_classes = np.unique(y_train)
    print(f"\nTraining classes present: {unique_classes}")

    if len(unique_classes) < 2:
        print("⚠ ERROR: Only one class present in training data!")
        return None

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        class_weight='balanced',  # Handle imbalance
        random_state=42,
        n_jobs=-1
    )

    print("\nTraining...")
    model.fit(X_train, y_train)

    # Evaluate
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)

    print(f"\n✓ Training accuracy: {train_score:.2%}")
    print(f"✓ Test accuracy: {test_score:.2%}")

    # Predictions
    y_pred = model.predict(X_test)

    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)

    # Only show classes that exist
    unique_test = np.unique(np.concatenate([y_test, y_pred]))
    target_names = ['Not Confirmed', 'Confirmed']

    if len(unique_test) == 2:
        print(classification_report(y_test, y_pred, target_names=target_names))
    else:
        print(classification_report(y_test, y_pred))

    print("\n" + "="*60)
    print("CONFUSION MATRIX")
    print("="*60)
    cm = confusion_matrix(y_test, y_pred)
    print(f"True Negatives:  {cm[0,0]}")
    if cm.shape[0] > 1:
        print(f"False Positives: {cm[0,1] if cm.shape[1] > 1 else 0}")
        print(f"False Negatives: {cm[1,0]}")
        print(f"True Positives:  {cm[1,1] if cm.shape[1] > 1 else 0}")

    # Feature importance
    print("\n" + "="*60)
    print("TOP FEATURES BY IMPORTANCE")
    print("="*60)
    importances = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    for idx, row in importances.iterrows():
        print(f"{row['feature']}: {row['importance']:.4f}")

    return model

# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    """Run the complete enhanced pipeline"""
    print("\n" + "="*60)
    print("ENHANCED MULTI-DATASET EXOPLANET CLASSIFICATION PIPELINE")
    print("="*60)
    print("Including: Kepler, K2, TESS TOI, and Confirmed Planets")
    print("="*60)

    try:
        # Step 1: Download data
        datasets = download_nasa_datasets()

        if not datasets:
            print("\n✗ No datasets available!")
            return None

        print(f"\n✓ Successfully loaded {len(datasets)} datasets:")
        for name in datasets.keys():
            print(f"  - {name}: {len(datasets[name])} rows")

        # Step 2: Integrate datasets
        combined_df = integrate_datasets(datasets)

        if combined_df is None:
            print("\n✗ Integration failed - insufficient data!")
            return None

        # Step 3: Clean data
        clean_df = clean_data(combined_df, missing_threshold=0.7)

        # Step 4: Prepare for ML
        X_train, X_test, y_train, y_test = prepare_for_ml(clean_df)

        # Step 5: Train model
        model = train_model(X_train, y_train, X_test, y_test)

        if model is None:
            print("\n✗ Model training failed!")
            return None

        # Save model automatically
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_path = os.path.join(DATA_DIR, f"exoplanet_model_{timestamp}.joblib")
        joblib.dump(model, model_path)
        print(f"\n💾 Model saved to: {model_path}")

        # Save feature information
        feature_info = {
            'feature_names': X_train.columns.tolist(),
            'training_date': timestamp,
            'datasets_used': list(datasets.keys()),
            'model_type': 'RandomForest',
            'test_accuracy': model.score(X_test, y_test)
        }

        info_path = os.path.join(DATA_DIR, f"model_info_{timestamp}.json")
        pd.Series(feature_info).to_json(info_path)
        print(f"💾 Model info saved to: {info_path}")

        print("\n" + "="*60)
        print("PIPELINE COMPLETE!")
        print("="*60)
        print(f"✓ Model trained successfully")
        print(f"✓ Total training samples: {len(X_train)}")
        print(f"✓ Features used: {len(X_train.columns)}")
        print(f"✓ Data saved in '{DATA_DIR}' directory")

        return model, X_train, X_test, y_train, y_test, model_path


    except Exception as e:
        print(f"\n✗ Pipeline error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# ==========================================
# RUN IT!
# ==========================================

if __name__ == "__main__":
    result = main()
    if result:
        model, X_train, X_test, y_train, y_test, model_path = result
        print("\n✓ All variables available for further analysis!")
        print(f"✓ Model ready for predictions!")
    else:
        print("\n⚠ Pipeline completed with errors. Check output above.")


ENHANCED MULTI-DATASET EXOPLANET CLASSIFICATION PIPELINE
Including: Kepler, K2, TESS TOI, and Confirmed Planets
DOWNLOADING NASA EXOPLANET DATASETS

  Description: Kepler Cumulative KOI table
✓ Downloaded 9564 rows, 153 columns

  Description: Confirmed Planets
✓ Downloaded 38952 rows, 354 columns

  Description: K2 Planets and Candidates
✓ Downloaded 4004 rows, 361 columns

  Description: TESS Objects of Interest
✗ Error downloading TESS TOI: No tables found
  Trying alternative TESS source...
✓ Downloaded TESS data from alternative source: 7703 rows
✓ Downloaded TESS TOI: 7703 rows, 109 columns

✓ Successfully loaded 4 datasets:
  - cumulative: 9564 rows
  - confirmed: 38952 rows
  - k2pandc: 4004 rows
  - tess_toi: 7703 rows

INTEGRATING DATASETS

--- Processing cumulative ---

Processing cumulative...
  Extracted 9564 samples (removed 0 without labels)
  Confirmed planets: 2746
  Not confirmed: 6818
✓ Added 9564 samples from cumulative

--- Processing confirmed ---

Processing con